                # BootstrapFinetune (Apple Silicon / MPS)

                Bootstrap successful traces into training data, then update model weights rather than only prompt state.

                **Use it when:** You control a trainable model and want to distill accepted DSPy traces into a reusable local adapter.

                **What compilation changes:** A PEFT LoRA adapter for Qwen2.5-0.5B-Instruct; the prompt remains separately inspectable.

                Important configuration in this benchmark:

                - DSPy BootstrapFinetune with a balanced-trace validation guard
- MPS selected on Apple Silicon; CPU remains an explicit fallback
- 18 full-profile training steps, batch size 1, LoRA rank 8, seed 42
- GPT-5.6 Sol teacher; Qwen2.5-0.5B student trained locally

                Every notebook loads the same 74-row dataset and frozen, pair-grouped
                train/validation/test membership before it can compile anything.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'bootstrap-finetune'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='bootstrap-finetune'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


                ## Compile shape

                This is the essential DSPy call used by the shared executable runner:

                ```python
                optimizer = BalancedBootstrapFinetune(
    metric=exact_match, train_kwargs=training_config,
    exclude_demos=True, num_threads=1, min_examples_per_class=2,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=sol_teacher,
)
                ```

                `compile` returns a program. The shared runner then evaluates that program on the
                untouched 20-example test split. The baseline has its own notebook; all other
                notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: bootstrap-finetune
task model: Qwen/Qwen2.5-0.5B-Instruct
final test accuracy: 35.0% (7/20)
accepted traces: human=17, AI=16
optimization time: 159.4s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/bootstrap-finetune.json
- canonical prompt: chapter06/results/final_prompts/bootstrap-finetune.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The corrected run accepted 17 human and 16 AI traces, then trained Qwen through DSPy on MPS. Its 35% final accuracy is a valid negative result, not the earlier one-class trace failure.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.